# Chapter 7 — Under the Hood: Compilation & QIR

**Learning objectives**
- Understand the Q# compilation pipeline: AST → HIR → FIR → RIR
- Understand what QIR is and why it matters for hardware targets
- Generate QIR with `qsharp.compile()` and inspect it
- Understand target profile constraints and their impact on algorithm design
- Visualize circuits with `qsharp.circuit()` and `qsharp_widgets.Circuit`
- Debug mid-circuit state with `DumpMachine`
- Understand the Azure Quantum submission pipeline

## Setup

In [ ]:
import qsharp
from qsharp_widgets import Circuit
print(f"qsharp {__import__("importlib.metadata", fromlist=["version"]).version("qsharp")}")

## 7.1 The Q# Compilation Pipeline

When you write Q# code, the compiler transforms it through several intermediate representations:

```
Q# source
    │
    ▼
AST  (Abstract Syntax Tree)
    │   Type checking, name resolution
    ▼
HIR  (High-level IR)
    │   Functor generation (Adjoint, Controlled)
    │   Partial evaluation of classical control
    ▼
FIR  (Functional IR)
    │   Qubit allocation, lifetime analysis
    │   Gate decomposition for target profile
    ▼
RIR  (Resolved IR)                ← Resource estimator reads here
    │   Gate-level, hardware-profile-checked
    ▼
QIR  (Quantum Intermediate Representation)
    │   LLVM-based IR
    ▼
Hardware target (simulator, QPU, Azure Quantum)
```

Each step enforces constraints and applies optimizations. The RIR is where the resource estimator does its counting.

## 7.2 What is QIR?

**QIR (Quantum Intermediate Representation)** is an open standard based on LLVM IR. It:

- Represents both classical and quantum computation in a single IR
- Enables hardware-agnostic compilation: compile once, target many QPUs
- Supports classical control flow (if/loops) that is executed on the control system
- Is the interchange format for Azure Quantum

Key QIR primitives:
- `%Qubit*` — opaque qubit type
- `%Result*` — opaque result type  
- `__quantum__qis__h__body(%Qubit*)` — single-qubit gate call
- `__quantum__qis__cnot__body(%Qubit*, %Qubit*)` — two-qubit gate
- `__quantum__qis__mz__body(%Qubit*, %Result*)` — measurement

## 7.3 Generating QIR

`qsharp.compile(entry_expr)` compiles the program to QIR and returns a `QSharpCompilation` object. The `str()` representation shows the LLVM IR text.

In [ ]:
# QIR generation requires the Base target profile
qsharp.init(target_profile=qsharp.TargetProfile.Base)
print("Initialized with TargetProfile.Base")

In [ ]:
%%qsharp

operation RandomNBits(N : Int) : Result[] {
    mutable results = [];
    for _ in 0..N-1 {
        use q = Qubit();
        H(q);
        let r = M(q);
        Reset(q);
        results += [r];
    }
    results
}

operation BellPair() : Result[] {
    use q = Qubit[2];
    H(q[0]);
    CNOT(q[0], q[1]);
    let r = MResetEachZ(q);
    r
}

In [ ]:
# Compile Bell pair to QIR
bell_qir = qsharp.compile("BellPair()")
qir_text = str(bell_qir)
print("QIR for BellPair() (truncated to 2000 chars):")
print(qir_text[:2000])

In [ ]:
# Count QIR gate calls
qir_lines = qir_text.split('\n')
gate_calls = [l.strip() for l in qir_lines if '__quantum__qis__' in l]
print(f"Total quantum gate/measurement calls in QIR: {len(gate_calls)}")
from collections import Counter
gate_types = Counter(l.split('__')[3].split('__')[0] for l in gate_calls if '__' in l)
print("Gate breakdown:")
for gate, count in gate_types.most_common():
    print(f"  {gate}: {count}")

## 7.4 Target Profile Constraints

The target profile determines what classical control flow is allowed at the hardware level:

| Profile | Mid-circuit measurement | Classical control on results | Use case |
|---------|------------------------|------------------------------|----------|
| `Base` | No | No | Gate-based QPUs, resource estimation |
| `Adaptive_RI` | Yes | Integer arithmetic | Quantinuum, mid-circuit feedback |
| `Unrestricted` | Yes | Full classical | Simulation only |

In [ ]:
# Show that Base profile rejects mid-circuit measurement control
from qsharp import QSharpError

try:
    qsharp.eval("""
        operation AdaptiveOp() : Result {
            use q = Qubit();
            H(q);
            let r = M(q);
            if r == One {  // Classical control on measurement — rejected by Base!
                X(q);
            }
            MResetZ(q)
        }
        AdaptiveOp()
    """)
except QSharpError as e:
    print(f"Base profile correctly rejects adaptive circuit:\n{str(e)[:400]}")

In [ ]:
# Switch to Adaptive_RI — now mid-circuit measurement control is allowed
qsharp.init(target_profile=qsharp.TargetProfile.Adaptive_RI)

result = qsharp.eval("""
    operation AdaptiveOp() : Result {
        use q = Qubit();
        H(q);
        let r = M(q);
        if r == One {
            X(q);  // Reset to |0⟩ conditionally
        }
        MResetZ(q)
    }
    AdaptiveOp()
""")
print(f"Adaptive operation result: {result} (always Zero after conditional correction)")

## 7.5 Circuit Visualization

`qsharp.circuit(entry_expr)` synthesizes a circuit diagram. `qsharp_widgets.Circuit` renders it as an interactive SVG.

Reference: `samples/notebooks/circuits.ipynb`

In [ ]:
# Circuit synthesis requires Base or Adaptive_RI profile (not Unrestricted)
qsharp.init(target_profile=qsharp.TargetProfile.Adaptive_RI)

In [ ]:
%%qsharp

operation GHZState(n : Int) : Result[] {
    use qs = Qubit[n];
    H(qs[0]);
    ApplyToEach(CNOT(qs[0], _), qs[1...]);
    let results = MeasureEachZ(qs);
    ResetAll(qs);
    results
}

In [ ]:
# Text circuit rendering
circuit_data = qsharp.circuit("GHZState(4)")
print(circuit_data)

In [ ]:
# Widget SVG rendering
Circuit(qsharp.circuit("GHZState(4)"))

In [ ]:
%%qsharp

operation QFT3(qs : Qubit[]) : Unit is Adj + Ctl {
    import Std.Math.PI;
    H(qs[0]);
    Controlled R1([qs[1]], (PI() / 2.0, qs[0]));
    Controlled R1([qs[2]], (PI() / 4.0, qs[0]));
    H(qs[1]);
    Controlled R1([qs[2]], (PI() / 2.0, qs[1]));
    H(qs[2]);
    SWAP(qs[0], qs[2]);
}

In [ ]:
# Circuit for QFT on 3 qubits — define a concrete wrapper so the qubit count is known
qsharp.eval("operation QFT3Demo() : Unit { use qs = Qubit[3]; QFT3(qs); }")
Circuit(qsharp.circuit("QFT3Demo()"))

In [ ]:
%%qsharp

// Circuit with conditional — cannot be statically synthesized, but can be simulated
operation TeleportState() : Result {
    use (msg, here, there) = (Qubit(), Qubit(), Qubit());
    // Prepare message qubit in |+⟩
    H(msg);
    // Create Bell pair between here and there
    H(here); CNOT(here, there);
    // Bell measurement on msg + here
    CNOT(msg, here);
    H(msg);
    let m1 = MResetZ(msg);
    let m2 = MResetZ(here);
    // Corrections
    if m2 == One { X(there); }
    if m1 == One { Z(there); }
    // Measure the teleported state
    MResetZ(there)
}

In [ ]:
from qsharp import CircuitGenerationMethod

# Static synthesis fails (conditional on measurement result)
try:
    Circuit(qsharp.circuit("TeleportState()"))
except Exception as e:
    print(f"Static synthesis fails: {str(e)[:200]}")
    print()
    # Simulation-based circuit shows one sample path
    print("Simulation-based circuit (one execution path):")
    Circuit(qsharp.circuit("TeleportState()", generation_method=CircuitGenerationMethod.Simulate))

## 7.6 Mid-Circuit Debugging with `DumpMachine`

`DumpMachine()` can be inserted at any point in a simulation to inspect the quantum state. This is the primary debugging tool in Q#.

In [ ]:
%%qsharp

import Std.Diagnostics.*;

operation DebugDemo() : Unit {
    use qs = Qubit[3];

    Message("--- Step 1: Initial state ---");
    DumpMachine();

    H(qs[0]);
    Message("--- Step 2: After H(qs[0]) ---");
    DumpMachine();

    CNOT(qs[0], qs[1]);
    Message("--- Step 3: After CNOT(qs[0], qs[1]) ---");
    DumpMachine();

    // Check an invariant using Fact
    // (would fail if state doesn't satisfy condition)

    Rz(0.5, qs[2]);
    Message("--- Step 4: After Rz(0.5, qs[2]) ---");
    DumpRegister(qs[2..2]);  // Only show qs[2]

    ResetAll(qs);
}

DebugDemo()

## 7.7 Runtime Capability Analysis

Different hardware targets have different capabilities. Q# categorizes operations by what they require:

| Requirement | Example | Hardware support |
|-------------|---------|------------------|
| Qubit allocation | `use q = Qubit()` | All |
| Gate application | `H(q)`, `CNOT(q1, q2)` | All |
| Measurement | `M(q)` | All |
| Classical branching on Result | `if M(q) == One { ... }` | Adaptive_RI+ |
| Integer arithmetic on results | `mutable x = 0; if r == One { x += 1 }` | Adaptive_RI |
| Repeat-until-success | `repeat { } until condition` | Unrestricted |

In [ ]:
# Demonstrate how the same algorithm behaves differently under profiles
profiles = [
    (qsharp.TargetProfile.Base, "Base"),
    (qsharp.TargetProfile.Adaptive_RI, "Adaptive_RI"),
    (qsharp.TargetProfile.Unrestricted, "Unrestricted"),
]

qqsharp_code = """
    operation SimpleAdaptive() : Int {
        use q = Qubit();
        H(q);
        let r = MResetZ(q);
        if r == One { return 1; } else { return 0; }
    }
    SimpleAdaptive()
"""

from qsharp import QSharpError
for profile, name in profiles:
    qsharp.init(target_profile=profile)
    try:
        result = qsharp.eval(qqsharp_code)
        print(f"{name:20s}: OK — result = {result}")
    except QSharpError as e:
        print(f"{name:20s}: REJECTED — {str(e)[:80]}")

## 7.8 Azure Quantum Submission Pipeline

To submit a Q# program to a real QPU via Azure Quantum:

1. **Initialize** with the appropriate target profile (usually `Base` or `Adaptive_RI`)
2. **Compile** with `qsharp.compile(entry_expr)` to generate QIR
3. **Connect** to Azure Quantum workspace
4. **Submit** to a target (QPU or emulator)
5. **Retrieve** results

Reference: `samples/notebooks/azure_submission.ipynb`

In [ ]:
# Initialize for Azure Quantum submission
qsharp.init(target_profile=qsharp.TargetProfile.Base)

In [ ]:
%%qsharp

operation RandomNBits(N : Int) : Result[] {
    mutable results = [];
    for _ in 0..N-1 {
        use q = Qubit();
        H(q);
        let r = M(q);
        Reset(q);
        results += [r];
    }
    results
}

In [ ]:
# Step 1: Compile to QIR
compilation = qsharp.compile("RandomNBits(4)")
print("Compiled to QIR successfully.")
print(f"QIR length: {len(str(compilation))} characters")

In [ ]:
# Step 2-5: Connect and submit (requires azure-quantum package and workspace credentials)
# Replace with your actual workspace details

print("Azure Quantum submission (stub — requires credentials):")
print()
azure_code = '''
import azure.quantum

workspace = azure.quantum.Workspace(
    subscription_id = "YOUR_SUBSCRIPTION_ID",
    resource_group  = "YOUR_RESOURCE_GROUP",
    name            = "YOUR_WORKSPACE_NAME",
    location        = "YOUR_LOCATION",  # e.g., "eastus"
)

# List available targets
targets = workspace.get_targets()
print("Available targets:", [t.name for t in targets])

# Submit to a simulator
target = workspace.get_targets("rigetti.sim.qvm")
job = target.submit(compilation, "my-random-bits-job", shots=100)
job.wait_until_completed()
results = job.get_results()
print("Results:", results)
'''
print(azure_code)

# Check if azure-quantum is installed
try:
    import azure.quantum
    print("azure-quantum is installed. Replace credentials above to run.")
except ImportError:
    print("To install: pip install azure-quantum")

## 7.9 How Target Profile Shapes Algorithm Design

Target profile constraints are not just technical limits — they drive algorithmic choices:

| Algorithm | Profile needed | Reason |
|-----------|---------------|--------|
| Gate-only QPE | Base | No mid-circuit control needed |
| Iterative QPE (semiclassical QFT) | Adaptive_RI | Feedback from mid-circuit measurement |
| Repeat-until-success state prep | Unrestricted | Loop depends on measurement outcome |
| Active QEC decoder | Adaptive_RI | Syndrome → correction in real time |
| Magic state distillation | Adaptive_RI | Post-selection on ancilla outcomes |

In [ ]:
# Demonstrate the semiclassical QFT (requires Adaptive_RI)
qsharp.init(target_profile=qsharp.TargetProfile.Adaptive_RI)

result = qsharp.eval("""
    import Std.Math.*;
    import Std.Convert.*;

    /// Semiclassical QFT: each qubit measured and classically controlled
    /// This requires Adaptive_RI (mid-circuit measurement + classical int arithmetic)
    operation SemiclassicalPhaseEst(nBits : Int) : Int {
        use (target, ctrl) = (Qubit(), Qubit());
        // Prepare target in eigenstate of Z (|1⟩ has eigenphase 0)
        X(target);

        mutable phase = 0;
        for k in nBits - 1..-1..0 {
            H(ctrl);
            // Controlled phase kick: ctrl acquires phase from 2^k applications
            let angle = 2.0 * PI() * IntAsDouble(1 <<< k);
            R1(angle, ctrl);  // Trivially 0 for Z eigenstate — this is just a demo structure

            // Classical feedback: rotate based on previously measured bits
            if phase &&& 1 == 1 {
                R1(-PI(), ctrl);
            }

            H(ctrl);
            let r = MResetZ(ctrl);
            if r == One {
                phase = phase ||| (1 <<< k);
            }
        }
        Reset(target);
        phase
    }

    SemiclassicalPhaseEst(4)
""")
print(f"Semiclassical phase estimate: {result} (runs on Adaptive_RI profile)")

## 7.10 Qiskit Integration: QIR via QSharpBackend

The `qiskit-qsharp` integration lets you compile Qiskit circuits to QIR for simulation or Azure Quantum submission. Reference: `samples/python_interop/qiskit.ipynb`

In [ ]:
# Generate QIR from a Qiskit circuit via QSharpBackend
try:
    from qiskit import QuantumCircuit
    from qsharp.interop.qiskit import QSharpBackend

    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])

    backend = QSharpBackend()
    # Explicitly specify Base profile — QIR generation requires Base or Adaptive_RI
    qir = backend.qir(qc, target_profile=qsharp.TargetProfile.Base)
    print("QIR from Qiskit Bell circuit (first 500 chars):")
    print(qir[:500])
except ImportError:
    print("qiskit not installed.")
    print("Install: pip install qiskit qiskit-qsharp")
    print()
    print("When installed, QSharpBackend.qir(circuit, target_profile=TargetProfile.Base)")
    print("returns LLVM IR text for submission to Azure Quantum or any QIR-compatible target.")

## Summary

- **Compilation pipeline**: Q# → AST → HIR → FIR → RIR → QIR, each stage enforcing invariants and applying optimizations
- **QIR**: LLVM-based, hardware-agnostic interchange format; gates are `__quantum__qis__*__body` LLVM calls
- **`qsharp.compile()`**: generates QIR for Azure Quantum submission; requires `TargetProfile.Base` or `Adaptive_RI`
- **Target profiles**: `Base` (gate-only), `Adaptive_RI` (mid-circuit feedback), `Unrestricted` (simulation only)
- **Circuit visualization**: `qsharp.circuit()` + `qsharp_widgets.Circuit` gives interactive SVG diagrams
- **Debugging**: `DumpMachine()` at any mid-circuit point; `DumpRegister()` for subsystems
- **Azure Quantum**: compile → connect workspace → `target.submit(compilation, shots=N)` → `job.get_results()`
- **Algorithm design**: profile constraints shape algorithm choices (iterative vs textbook QPE, RUS loops, active QEC)